### Complete Order (Delay)
An objective to minimize delay can be considered to induce a complete order upon two separation identical aircraft $i$ and $j$ if $r_i \leq r_j$ and $b_i \leq b_j$. That is, the makespan of any partial sequence $S$, which schedules aircraft $i$ before aircraft $j$ will result in a per-aircraft delay no worse than in $S^\prime$ (schedulling $j$ after $i$); excluding aircraft $i$ and $j$.


In [52]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Encode and enforce the rule premises; total order on release/base times and that i and j are δ-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],   ctx.b["i"] <= ctx.b["j"],   *ctx.separation_equivalence("i", "j"),
]



# Encode the claim; no aircraft is more delayed in S1 than S2 (aside i and j).
claim = And(*[
    S1.delay[x] <= S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(res)



# Encode the claim; total delay in S1 ≤ total delay in S2.
claim = Sum([S1.delay[x] for x in ctx.aircraft]) <= Sum([S2.delay[x] for x in ctx.aircraft])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(res)


Verified
Verified


### Complete Order (Makespan)
An objective to minimize makespan can be considered to induce a complete order upon two separation identical aircraft $i$ and $j$ if $r_i \leq r_j$. That is, the makespan of any partial sequence $S$, which schedules aircraft $i$ before aircraft $j$ will result in a makespan no worse than in $S^\prime$ (schedulling $j$ after $i$); therefore aircraft $i$ should always be scheduled before aircraft $j$.

In [36]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Encode and enforce the rule premises; total order on release times and that i and j are δ-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],   *ctx.separation_equivalence("i", "j"),
]

# Encode the claim; makespan in S1 is less than in S2.
claim = S1.makespan <= S2.makespan

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(res)


Verified


### Complete Order (Time Window Feasibility)


In [39]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Encode and enforce the rule premises; total order on release times and that i and j are δ-equivalent, and that S2 is feasible.
premises = [
    ctx.r["i"] <= ctx.r["j"],               ctx.lt["i"] <= ctx.lt["j"],
    *ctx.separation_equivalence("i", "j"),  *S2.time_window_feasible,
]

# Encode the claim; no aircraft hits it's time window in S2 but not in S1.
claim = And(*[
    Implies(S2.window_violation[x], S1.window_violation[x])
    for x in ctx.aircraft
])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(res)


Verified
